# Preprocessing — Eksperimen 5

**Alur:** load → split → delexicalization → belief span → format per turn → tokenisasi → vocabulary → word ke index → padding → tensor.

Dua keputusan penting di pipeline ini:
1. **Delexicalization word-boundary** (`\b...\b`) — nilai entitas tak mencemari kata lain (mis. `asked` tidak jadi `NAME_SLOTed`).
2. **`min_freq=2`** — kata yang muncul sekali (umumnya typo) dibuang jadi OOV, memaksa mekanisme *copy* TSCP menyalin nilai langka langsung dari input (keunggulan OOV pada paper).

Setiap tahap (3–11) mengekspor 3 file train/val/test ke `data/processed/eksp5/`.

## 1. Konfigurasi

Rasio split, token khusus (`<pad>`, `<sos>`, `<eos>`, `<unk>`, `<Inf>`, `<Req>`, token slot), dan daftar slot informable.

In [160]:
import json, re, random
from pathlib import Path
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"

# Split 3:1:1 sesuai paper
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.6, 0.2, 0.2
SEED = 42

# Special tokens
PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = "<pad>", "<sos>", "<eos>", "<unk>"
INF_OPEN, INF_CLOSE, REQ_OPEN, REQ_CLOSE = "<Inf>", "</Inf>", "<Req>", "</Req>"
SLOT_TOKENS = ["NAME_SLOT", "ADDRESS_SLOT", "PHONE_SLOT", "POSTCODE_SLOT",
               "FOOD_SLOT", "AREA_SLOT", "PRICERANGE_SLOT"]
SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN,
                  INF_OPEN, INF_CLOSE, REQ_OPEN, REQ_CLOSE] + SLOT_TOKENS

# Slot CamRest676
INFORMABLE_SLOTS = ["food", "area", "pricerange"]
DB_FIELD_TO_SLOT = {"name": "NAME_SLOT", "address": "ADDRESS_SLOT", "phone": "PHONE_SLOT",
                    "postcode": "POSTCODE_SLOT", "food": "FOOD_SLOT", "area": "AREA_SLOT",
                    "pricerange": "PRICERANGE_SLOT"}

print("Root repo:", ROOT)
print("Data dir :", DATA_DIR)
print(f"Special tokens ({len(SPECIAL_TOKENS)}):", SPECIAL_TOKENS)

# ===== Setup ekspor output PER TAHAP (train/val/test) =====
PROCESSED_DIR = ROOT / "data" / "processed" / "eksp5"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def _fmt_json(obj, indent=1, level=0):
    """Pretty-print: dict turun baris per key; list skalar (angka/teks) INLINE 1 baris;
    list-of-list (matriks) -> tiap sub-list 1 baris. Biar mudah dibaca, tak terlalu vertikal."""
    pad, pad1 = " " * (indent * level), " " * (indent * (level + 1))
    if isinstance(obj, dict):
        if not obj:
            return "{}"
        items = [f'{pad1}{json.dumps(k, ensure_ascii=False)}: {_fmt_json(v, indent, level + 1)}'
                 for k, v in obj.items()]
        return "{\n" + ",\n".join(items) + "\n" + pad + "}"
    if isinstance(obj, list):
        if not obj:
            return "[]"
        if all(not isinstance(e, (dict, list)) for e in obj):  # list skalar -> inline
            return "[" + ", ".join(json.dumps(e, ensure_ascii=False) for e in obj) + "]"
        items = [f'{pad1}{_fmt_json(e, indent, level + 1)}' for e in obj]
        return "[\n" + ",\n".join(items) + "\n" + pad + "]"
    return json.dumps(obj, ensure_ascii=False)


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(_fmt_json(obj) + "\n")
    return path


def export_splits(stage_tag, splits, fmt="json"):
    """Ekspor 3 file (train/val/test) untuk satu tahapan.

    splits: dict {"train": ..., "val": ..., "test": ...}
    """
    paths = {}
    for name, obj in splits.items():
        p = PROCESSED_DIR / f"{stage_tag}_{name}.{fmt}"
        save_json(obj, p)
        paths[name] = p
    print("  [EXPORT] " + stage_tag + ": " +
          ", ".join(f"{n}->{pp.name}" for n, pp in paths.items()))
    return paths


print("Folder output per-tahap:", PROCESSED_DIR)


Root repo: e:\coding bebas\restorant-asistent
Data dir : e:\coding bebas\restorant-asistent\data
Special tokens (15): ['<pad>', '<sos>', '<eos>', '<unk>', '<Inf>', '</Inf>', '<Req>', '</Req>', 'NAME_SLOT', 'ADDRESS_SLOT', 'PHONE_SLOT', 'POSTCODE_SLOT', 'FOOD_SLOT', 'AREA_SLOT', 'PRICERANGE_SLOT']
Folder output per-tahap: e:\coding bebas\restorant-asistent\data\processed\eksp5


## 2. Load Data Mentah

Baca `CamRest676.json` (676 dialog percakapan) dan `CamRestDB.json` (database restoran) — sumber semua tahap berikutnya.

In [161]:
def load_raw_data():
    with open(DATA_DIR / "CamRest676.json") as f:
        dialogues = json.load(f)
    with open(DATA_DIR / "CamRestDB.json") as f:
        database = json.load(f)
    return dialogues, database

dialogues, database = load_raw_data()

turn = dialogues[0]["dial"][0]
print(f"Total dialog : {len(dialogues)}")
print(f"Total KB     : {len(database)} restoran\n")
print("Contoh 1 turn mentah:")
print("  usr:", turn["usr"]["transcript"])
print("  sys:", turn["sys"]["sent"])
print("\nContoh entry KB:")
print(" ", database[0])

Total dialog : 676
Total KB     : 110 restoran

Contoh 1 turn mentah:
  usr: I need to find an expensive restauant that's in the south section of the city.
  sys: There are several restaurants in the south part of town that serve expensive food. Do you have a cuisine preference?

Contoh entry KB:
  {'address': 'Regent Street City Centre', 'area': 'centre', 'food': 'italian', 'location': '52.20103,0.126023', 'phone': '01223 323737', 'pricerange': 'cheap', 'postcode': 'C.B 2, 1 A.B', 'type': 'restaurant', 'id': '19210', 'name': 'pizza hut city centre'}


In [162]:
# ===== Sampel jangkar untuk demo tiap tahap: dialog #520, turn 1 =====
SAMPLE_DIALOGUE_ID, SAMPLE_TI = 520, 1
SAMPLE_DIAL = next(d for d in dialogues if d.get("dialogue_id") == SAMPLE_DIALOGUE_ID)
SAMPLE_TURN = SAMPLE_DIAL["dial"][SAMPLE_TI]


def demo_sample(stage, inp, proses, out):
    print("=" * 90)
    print(f"[SAMPLE dialog #{SAMPLE_DIALOGUE_ID} turn {SAMPLE_TI}]", stage)
    print("=" * 90)
    print("INPUT  :", inp)
    print("PROSES :", proses)
    print("OUTPUT :", out)


In [163]:
# Tahap 2 - Data mentah  (AWAL RANTAI: sample = system response turn terpilih)
S_TURN = SAMPLE_TURN
S_RESP_RAW = S_TURN["sys"]["sent"]
demo_sample("Tahap 2 - Data mentah (per turn)", "CamRest676.json (676 dialog)",
            "ambil user utterance + system response tiap turn",
            f"USR {S_TURN['usr']['transcript']!r}  |  SYS {S_RESP_RAW!r}")

total_turn = sum(len(d["dial"]) for d in dialogues)
print(f"\n  Dimensi: {len(dialogues)} dialog -> {total_turn} turn (belum di-split)")


[SAMPLE dialog #520 turn 1] Tahap 2 - Data mentah (per turn)
INPUT  : CamRest676.json (676 dialog)
PROSES : ambil user utterance + system response tiap turn
OUTPUT : USR 'What kind of food do they serve?'  |  SYS 'la margherita is an italian restaurant in the area of west in the cheap price range. '

  Dimensi: 676 dialog -> 2744 turn (belum di-split)


## 3. Split Data (3:1:1)

Bagi dialog jadi train/val/test rasio 3:1:1 (mengikuti paper). Split **per-dialog** dengan `seed` tetap: reproducible dan bebas leakage antar-turn.

In [164]:
def split_data(dialogues, seed=SEED):
    random.seed(seed)
    idx = list(range(len(dialogues)))
    random.shuffle(idx)
    n = len(dialogues)
    n_train, n_val = int(n * TRAIN_RATIO), int(n * VAL_RATIO)
    train = [dialogues[i] for i in idx[:n_train]]
    val = [dialogues[i] for i in idx[n_train:n_train + n_val]]
    test = [dialogues[i] for i in idx[n_train + n_val:]]
    return train, val, test

train_dial, val_dial, test_dial = split_data(dialogues)

In [165]:
# Tahap 3 - Split data
_which = "train" if SAMPLE_DIAL in train_dial else ("val" if SAMPLE_DIAL in val_dial else "test")
demo_sample("Tahap 3 - Split 3:1:1", "676 dialog",
            "acak (seed 42) lalu bagi 60/20/20 -> train/val/test",
            f"dialog sample masuk ke split: {_which}")

print(f"\n  Dimensi: train={len(train_dial)} | val={len(val_dial)} | test={len(test_dial)} dialog")

# EKSPOR Tahap 3 (nested: dialog utuh hasil split)
export_splits("03_split", {"train": train_dial, "val": val_dial, "test": test_dial})


[SAMPLE dialog #520 turn 1] Tahap 3 - Split 3:1:1
INPUT  : 676 dialog
PROSES : acak (seed 42) lalu bagi 60/20/20 -> train/val/test
OUTPUT : dialog sample masuk ke split: train

  Dimensi: train=405 | val=135 | test=136 dialog
  [EXPORT] 03_split: train->03_split_train.json, val->03_split_val.json, test->03_split_test.json


{'train': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/03_split_train.json'),
 'val': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/03_split_val.json'),
 'test': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/03_split_test.json')}

## 4. Delexicalization

Ganti nilai spesifik restoran (nama, alamat, telepon, ...) di **response sistem** dengan placeholder generik (`NAME_SLOT`, `ADDRESS_SLOT`, ...). Model belajar pola kalimat, bukan menghafal entitas; nilai asli ditempel lagi saat inference (lexicalization).

Hanya response yang di-delex — input user & belief span dibiarkan asli. Nilai dicocokkan **word-boundary** dan diurutkan terpanjang dulu (frasa panjang menang atas frasa pendek).

In [166]:
def collect_slot_values(database):
    """Kumpulkan pasangan (value, slot) unik dari DB, diurutkan value terpanjang dulu.

    Urutan terpanjang-dulu mencegah frasa pendek menimpa frasa panjang
    (mis. 'city centre' harus diproses sebelum 'centre').
    """
    pairs, seen = [], set()
    for entry in database:
        for field, slot in DB_FIELD_TO_SLOT.items():
            val = str(entry.get(field, "")).lower().strip()
            if val and (val, slot) not in seen:
                seen.add((val, slot))
                pairs.append((val, slot))
    pairs.sort(key=lambda p: len(p[0]), reverse=True)
    return pairs


_WB = chr(92) + "b"  # penanda word-boundary regex, ditulis via chr(92) agar aman


def delexicalize_response(response, slot_value_pairs):
    """Ganti nilai entitas di response dengan placeholder slot.

    Semua slot memakai word-boundary agar nilai tidak mencemari kata lain
    (mis. 'asked' tidak menjadi 'NAME_SLOTed').
    """
    delex = response.lower()
    for val, slot in slot_value_pairs:
        delex = re.sub(_WB + re.escape(val) + _WB, slot, delex)
    return delex


SLOT_VALUE_PAIRS = collect_slot_values(database)


def build_delex_dialogue(dialogue):
    """Bentuk NESTED: salin dialog, tambahkan sys.sent_delex per turn.

    Hanya respons sistem (R_t) yang di-delex; user & slu dibiarkan asli.
    """
    return {
        "dialogue_id": dialogue.get("dialogue_id", ""),
        "dial": [
            {
                "turn": ti,
                "usr": {"transcript": turn["usr"]["transcript"], "slu": turn["usr"]["slu"]},
                "sys": {
                    "sent": turn["sys"]["sent"],
                    "sent_delex": delexicalize_response(
                        turn["sys"]["sent"].lower().strip(), SLOT_VALUE_PAIRS
                    ),
                },
            }
            for ti, turn in enumerate(dialogue["dial"])
        ],
    }


def delex_split(dials):
    return [build_delex_dialogue(d) for d in dials]


In [167]:
# Tahap 4 - Delexicalization  (INPUT = output Tahap 3: dialog hasil split; tetap NESTED)
train_delex = delex_split(train_dial)
val_delex = delex_split(val_dial)
test_delex = delex_split(test_dial)

S_RESP_DELEX = delexicalize_response(S_RESP_RAW.lower().strip(), SLOT_VALUE_PAIRS)
demo_sample("Tahap 4 - Delexicalization",
            f"[dari Tahap 3] {S_RESP_RAW!r}",
            "ganti nilai entitas -> placeholder slot (word-boundary); disimpan di sys.sent_delex (tetap nested)",
            S_RESP_DELEX)

# EKSPOR Tahap 4 (nested: dialog + sys.sent_delex)
export_splits("04_delex", {"train": train_delex, "val": val_delex, "test": test_delex})


[SAMPLE dialog #520 turn 1] Tahap 4 - Delexicalization
INPUT  : [dari Tahap 3] 'la margherita is an italian restaurant in the area of west in the cheap price range. '
PROSES : ganti nilai entitas -> placeholder slot (word-boundary); disimpan di sys.sent_delex (tetap nested)
OUTPUT : NAME_SLOT is an FOOD_SLOT restaurant in the area of AREA_SLOT in the PRICERANGE_SLOT price range.
  [EXPORT] 04_delex: train->04_delex_train.json, val->04_delex_val.json, test->04_delex_test.json


{'train': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/04_delex_train.json'),
 'val': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/04_delex_val.json'),
 'test': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/04_delex_test.json')}

## 5. Belief Span (bspan)

Ringkas kebutuhan user per turn jadi satu teks: `<Inf> value ; value </Inf> <Req> slot ; slot </Req>`.
- **Informable** (`<Inf>`): nilai kriteria pencarian (mis. `italian`).
- **Requestable** (`<Req>`): nama slot yang diminta user (mis. `address`).

Menggantikan intent + slot classifier dengan satu representasi teks yang bisa di-generate model.

In [168]:
def construct_bspan(slu_annotations):
    informable, requestable = [], []
    for slu in slu_annotations:
        act = slu["act"]
        for pair in slu["slots"]:
            if act == "inform":
                name, value = pair[0], pair[1]
                if value != "dontcare" and name in INFORMABLE_SLOTS and value.lower() not in informable:
                    informable.append(value.lower())
            elif act == "request":
                req = pair[1] if pair[0] == "slot" else pair[0]
                if req.lower() not in requestable:
                    requestable.append(req.lower())
    return (f"{INF_OPEN} {' ; '.join(informable)} {INF_CLOSE} "
            f"{REQ_OPEN} {' ; '.join(requestable)} {REQ_CLOSE}")


def build_bspan_dialogue(dialogue_delex):
    """INPUT = dialog hasil Tahap 4 (nested). Tambahkan usr.bspan per turn. Tetap NESTED."""
    out = {"dialogue_id": dialogue_delex["dialogue_id"], "dial": []}
    for turn in dialogue_delex["dial"]:
        out["dial"].append({
            "turn": turn["turn"],
            "usr": {
                "transcript": turn["usr"]["transcript"],
                "slu": turn["usr"]["slu"],
                "bspan": construct_bspan(turn["usr"]["slu"]),
            },
            "sys": {"sent": turn["sys"]["sent"], "sent_delex": turn["sys"]["sent_delex"]},
        })
    return out


def bspan_split(dials_delex):
    return [build_bspan_dialogue(d) for d in dials_delex]


In [169]:
# Tahap 5 - Belief span  (INPUT = output Tahap 4 nested; cabang paralel dari anotasi SLU user)
train_bspan = bspan_split(train_delex)
val_bspan = bspan_split(val_delex)
test_bspan = bspan_split(test_delex)

S_BSPAN = construct_bspan(S_TURN["usr"]["slu"])
demo_sample("Tahap 5 - Belief span",
            f"[SLU user turn ini] {S_TURN['usr']['slu']}",
            "informable = value (act inform); requestable = nama slot (act request) -> <Inf>..</Inf> <Req>..</Req>; disimpan di usr.bspan",
            S_BSPAN)

# EKSPOR Tahap 5 (nested: dialog + usr.bspan + sys.sent_delex) -- tahap TERAKHIR yang masih nested
export_splits("05_bspan", {"train": train_bspan, "val": val_bspan, "test": test_bspan})


[SAMPLE dialog #520 turn 1] Tahap 5 - Belief span
INPUT  : [SLU user turn ini] [{'act': 'request', 'slots': [['slot', 'food']]}, {'act': 'inform', 'slots': [['pricerange', 'cheap']]}, {'act': 'inform', 'slots': [['area', 'west']]}]
PROSES : informable = value (act inform); requestable = nama slot (act request) -> <Inf>..</Inf> <Req>..</Req>; disimpan di usr.bspan
OUTPUT : <Inf> cheap ; west </Inf> <Req> food </Req>
  [EXPORT] 05_bspan: train->05_bspan_train.json, val->05_bspan_val.json, test->05_bspan_test.json


{'train': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/05_bspan_train.json'),
 'val': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/05_bspan_val.json'),
 'test': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/05_bspan_test.json')}

## 6. Format Sample per Turn

Susun tiap turn jadi pasangan input→target sesuai rumus Sequicity (`B_t = seq2seq(B_{t-1} R_{t-1} U_t)`):
- **input**: bspan + response turn sebelumnya + ucapan user sekarang
- **target_bspan**: yang diprediksi decoder tahap 1
- **target_response**: yang diprediksi decoder tahap 2

Di sini 676 dialog mengembang jadi ribuan sample per-turn — bentuk data berubah **nested → flat**.

In [170]:
def process_dialogue(dialogue_bspan):
    """INPUT = dialog Tahap 5 (nested; sudah punya usr.bspan + sys.sent_delex).

    OUTPUT = list sample per turn (FLAT): {input=B_(t-1)R_(t-1)U_t, target_bspan, target_response}.
    B_0 dan R_0 = empty string (paper).
    """
    processed, prev_bspan, prev_response = [], "", ""
    for turn in dialogue_bspan["dial"]:
        user = turn["usr"]["transcript"].lower().strip()
        bspan = turn["usr"]["bspan"]
        response = turn["sys"]["sent_delex"]
        parts = [p for p in (prev_bspan, prev_response) if p] + [user]
        processed.append({
            "input": " ".join(parts),
            "target_bspan": bspan,
            "target_response": response,
            "dialogue_id": dialogue_bspan["dialogue_id"],
            "turn": turn["turn"],
        })
        prev_bspan, prev_response = bspan, response
    return processed


def process_all(dialogues_bspan):
    return [s for d in dialogues_bspan for s in process_dialogue(d)]


train_samples = process_all(train_bspan)
val_samples = process_all(val_bspan)
test_samples = process_all(test_bspan)


In [171]:
# Tahap 6 - Format sample per turn  (INPUT = Tahap 5 nested -> FLATTEN jadi sample per turn)
S_SAMPLE = process_dialogue(build_bspan_dialogue(build_delex_dialogue(SAMPLE_DIAL)))[SAMPLE_TI]
demo_sample("Tahap 6 - Format sample per turn",
            f"bspan[Tahap5]={S_BSPAN}  ;  target_response[Tahap4]={S_RESP_DELEX!r}  ;  user={S_TURN['usr']['transcript']!r}",
            "susun jadi {input=B_(t-1)R_(t-1)U_t, target_bspan, target_response} -- bentuk data JADI FLAT",
            f"input={S_SAMPLE['input']!r}\n          target_bspan={S_SAMPLE['target_bspan']}\n          target_response={S_SAMPLE['target_response']!r}")

print(f"\n  Dimensi: {len(train_samples) + len(val_samples) + len(test_samples)} sample turn "
      f"(train={len(train_samples)} | val={len(val_samples)} | test={len(test_samples)})")

# EKSPOR Tahap 6 (FLAT: sample per turn)
export_splits("06_format", {"train": train_samples, "val": val_samples, "test": test_samples})


[SAMPLE dialog #520 turn 1] Tahap 6 - Format sample per turn
INPUT  : bspan[Tahap5]=<Inf> cheap ; west </Inf> <Req> food </Req>  ;  target_response[Tahap4]='NAME_SLOT is an FOOD_SLOT restaurant in the area of AREA_SLOT in the PRICERANGE_SLOT price range.'  ;  user='What kind of food do they serve?'
PROSES : susun jadi {input=B_(t-1)R_(t-1)U_t, target_bspan, target_response} -- bentuk data JADI FLAT
OUTPUT : input='<Inf> cheap ; west </Inf> <Req>  </Req> how about NAME_SLOT? what kind of food do they serve?'
          target_bspan=<Inf> cheap ; west </Inf> <Req> food </Req>
          target_response='NAME_SLOT is an FOOD_SLOT restaurant in the area of AREA_SLOT in the PRICERANGE_SLOT price range.'

  Dimensi: 2744 sample turn (train=1635 | val=553 | test=556)
  [EXPORT] 06_format: train->06_format_train.json, val->06_format_val.json, test->06_format_test.json


{'train': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/06_format_train.json'),
 'val': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/06_format_val.json'),
 'test': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/06_format_test.json')}

## 7. Tokenisasi

Pecah teks jadi token per kata dan pisahkan tanda baca. Token khusus (`<Inf>`, `NAME_SLOT`, ...) dilindungi agar tidak ikut terpecah. Bspan & response ditambah penanda `<sos>`/`<eos>`.

In [172]:
def tokenize(text, protected_tokens):
    sorted_protected = sorted(protected_tokens, key=len, reverse=True)
    pattern_protected = "|".join(re.escape(tok) for tok in sorted_protected)

    segments = re.split(f"({pattern_protected})", text)

    result_tokens = []
    for segment in segments:
        if segment in protected_tokens:
            # Token spesial tidak di tokenisasi
            result_tokens.append(segment)
        elif segment.strip():
            # Teks biasa di tokenisasi
            cleaned = re.sub(r"([.,!?;:'\"\(\)])", r" \1 ", segment)
            cleaned = re.sub(r"\s+", " ", cleaned).strip()
            result_tokens.extend(cleaned.split())

    return result_tokens


def build_tokenized_record(sample):
    tokens_bspan_core = tokenize(sample["target_bspan"], SPECIAL_TOKENS)
    tokens_response_core = tokenize(sample["target_response"], SPECIAL_TOKENS)

    return {
        "tokens_input": tokenize(sample["input"], SPECIAL_TOKENS),
        "tokens_bspan": [SOS_TOKEN] + tokens_bspan_core + [EOS_TOKEN],
        "tokens_response": [SOS_TOKEN] + tokens_response_core + [EOS_TOKEN],
        "dialogue_id": sample["dialogue_id"],
        "turn": sample["turn"],
    }


def tokenize_dataset(samples):
    return [build_tokenized_record(s) for s in samples]


train_tokenized = tokenize_dataset(train_samples)
val_tokenized = tokenize_dataset(val_samples)
test_tokenized = tokenize_dataset(test_samples)

In [173]:
# Tahap 7 - Tokenisasi  (INPUT = output Tahap 6: target_response)
S_TOK = build_tokenized_record(S_SAMPLE)
demo_sample("Tahap 7 - Tokenisasi (response)",
            f"[dari Tahap 6] target_response = {S_SAMPLE['target_response']!r}",
            "pecah jadi token; token khusus dilindungi; tanda baca dipisah; +<sos>/<eos>",
            f"{S_TOK['tokens_response']}  ({len(S_TOK['tokens_response'])} token)")

# EKSPOR Tahap 7 (FLAT: token per sample)
export_splits("07_tokenized", {"train": train_tokenized, "val": val_tokenized, "test": test_tokenized})


[SAMPLE dialog #520 turn 1] Tahap 7 - Tokenisasi (response)
INPUT  : [dari Tahap 6] target_response = 'NAME_SLOT is an FOOD_SLOT restaurant in the area of AREA_SLOT in the PRICERANGE_SLOT price range.'
PROSES : pecah jadi token; token khusus dilindungi; tanda baca dipisah; +<sos>/<eos>
OUTPUT : ['<sos>', 'NAME_SLOT', 'is', 'an', 'FOOD_SLOT', 'restaurant', 'in', 'the', 'area', 'of', 'AREA_SLOT', 'in', 'the', 'PRICERANGE_SLOT', 'price', 'range', '.', '<eos>']  (18 token)
  [EXPORT] 07_tokenized: train->07_tokenized_train.json, val->07_tokenized_val.json, test->07_tokenized_test.json


{'train': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/07_tokenized_train.json'),
 'val': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/07_tokenized_val.json'),
 'test': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/07_tokenized_test.json')}

## 8. Vocabulary

Bangun kamus token→index **hanya dari data training** (cegah leakage dari val/test). Token khusus menempati index awal, sisanya diurutkan per frekuensi.

`min_freq=2`: kata yang muncul sekali (umumnya typo) dibuang → jadi **OOV**, sehingga model dipaksa belajar meng-*copy* nilai langka dari input alih-alih menghafalnya.

In [174]:
def build_vocabulary(tokenized_samples, min_freq=2, max_vocab_size=800):
    """Bangun word2idx dari training set.

    - Token khusus (special tokens) selalu masuk, menempati index awal.
    - Kata dengan frekuensi < min_freq dibuang (jadi OOV) untuk melatih copy.
    - Sisanya diurutkan frekuensi menurun, dipotong pada max_vocab_size.
    """
    counter = Counter()
    for s in tokenized_samples:
        counter.update(s["tokens_input"])
        counter.update(s["tokens_bspan"])
        counter.update(s["tokens_response"])

    word2idx = {tok: i for i, tok in enumerate(SPECIAL_TOKENS)}
    sorted_words = sorted(counter.items(), key=lambda x: (-x[1], x[0]))

    n_dropped_rare = 0
    for word, freq in sorted_words:
        if word in word2idx:
            continue
        if freq < min_freq:
            n_dropped_rare += 1
            continue
        if len(word2idx) >= max_vocab_size:
            break
        word2idx[word] = len(word2idx)

    idx2word = {i: w for w, i in word2idx.items()}

    print(f"Vocab size: {len(word2idx)} (min_freq={min_freq}, max={max_vocab_size})")
    print(f"Token langka dibuang jadi OOV: {n_dropped_rare}")
    print("12 entri pertama:", list(word2idx.items())[:12])
    return word2idx, idx2word


word2idx, idx2word = build_vocabulary(train_tokenized, min_freq=2, max_vocab_size=800)

Vocab size: 613 (min_freq=2, max=800)
Token langka dibuang jadi OOV: 137
12 entri pertama: [('<pad>', 0), ('<sos>', 1), ('<eos>', 2), ('<unk>', 3), ('<Inf>', 4), ('</Inf>', 5), ('<Req>', 6), ('</Req>', 7), ('NAME_SLOT', 8), ('ADDRESS_SLOT', 9), ('PHONE_SLOT', 10), ('POSTCODE_SLOT', 11)]


In [175]:
# Tahap 8 - Vocabulary  (INPUT = token response dari Tahap 7)
_toks = S_TOK["tokens_response"][:8]
_map = {t: word2idx.get(t, word2idx[UNK_TOKEN]) for t in _toks}
demo_sample("Tahap 8 - Vocabulary (token -> index)",
            f"[token response dari Tahap 7] {_toks}",
            "bangun word2idx dari TRAIN (min_freq=2); kata freq<2 tidak dapat index (OOV/<unk>)",
            _map)

print(f"\n  Dimensi: ukuran vocabulary = {len(word2idx)} token")

# EKSPOR Tahap 8 (vocabulary = artefak TUNGGAL, dibangun dari TRAIN saja -> tidak displit train/val/test)
save_json(word2idx, PROCESSED_DIR / "08_vocab_word2idx.json")
save_json({str(k): v for k, v in idx2word.items()}, PROCESSED_DIR / "08_vocab_idx2word.json")
print(f"  [EXPORT] 08_vocab: word2idx ({len(word2idx)} token) -> 08_vocab_word2idx.json + 08_vocab_idx2word.json")
print("           (tunggal, bukan 3-split: vocab HANYA dari train demi mencegah data leakage)")


[SAMPLE dialog #520 turn 1] Tahap 8 - Vocabulary (token -> index)
INPUT  : [token response dari Tahap 7] ['<sos>', 'NAME_SLOT', 'is', 'an', 'FOOD_SLOT', 'restaurant', 'in', 'the']
PROSES : bangun word2idx dari TRAIN (min_freq=2); kata freq<2 tidak dapat index (OOV/<unk>)
OUTPUT : {'<sos>': 1, 'NAME_SLOT': 8, 'is': 18, 'an': 75, 'FOOD_SLOT': 12, 'restaurant': 32, 'in': 21, 'the': 17}

  Dimensi: ukuran vocabulary = 613 token
  [EXPORT] 08_vocab: word2idx (613 token) -> 08_vocab_word2idx.json + 08_vocab_idx2word.json
           (tunggal, bukan 3-split: vocab HANYA dari train demi mencegah data leakage)


## 9. Word ke Index

Ubah token jadi angka index sesuai vocabulary. Token di luar vocab (OOV) diberi index sementara ≥ `vocab_size` agar bisa ditangani mekanisme copy (CopyNet), bukan sekadar dibuang.

In [176]:
def tokens_to_indices_copynet(tokens, word2idx):
    unk_idx = word2idx[UNK_TOKEN]
    vocab_size = len(word2idx)
    
    indices = []
    oov_words = []
    oov_map = {} # Memetakan kata OOV ke index sementara (vocab_size, vocab_size+1, dst)
    
    for t in tokens:
        if t in word2idx:
            indices.append(word2idx[t])
        else:
            # Jika kata tidak ada di vocab, beri index khusus untuk mekanisme CopyNet
            if t not in oov_map:
                oov_map[t] = vocab_size + len(oov_words)
                oov_words.append(t)
            indices.append(oov_map[t])
            
    return indices, oov_words

def build_indexed_record(tokenized_sample, word2idx):
    input_indices, input_oov = tokens_to_indices_copynet(tokenized_sample["tokens_input"], word2idx)
    bspan_indices, _ = tokens_to_indices_copynet(tokenized_sample["tokens_bspan"], word2idx)
    response_indices, _ = tokens_to_indices_copynet(tokenized_sample["tokens_response"], word2idx)
    
    return {
        "input_indices": input_indices,
        "input_oov": input_oov, # Menyimpan list kata OOV asli dari input
        "bspan_indices": bspan_indices,
        "response_indices": response_indices,
        "dialogue_id": tokenized_sample["dialogue_id"],
        "turn": tokenized_sample["turn"],
    }

def indexize_dataset(tokenized_samples, word2idx):
    return [build_indexed_record(s, word2idx) for s in tokenized_samples]

train_indexed = indexize_dataset(train_tokenized, word2idx)
val_indexed = indexize_dataset(val_tokenized, word2idx)
test_indexed = indexize_dataset(test_tokenized, word2idx)

In [177]:
# Tahap 9 - Word ke index  (INPUT = output Tahap 7: tokens_response)
S_IDX = build_indexed_record(S_TOK, word2idx)
_V = len(word2idx)
_resp_ids, _resp_oov = tokens_to_indices_copynet(S_TOK["tokens_response"], word2idx)
demo_sample("Tahap 9 - Word ke index (response)",
            f"[dari Tahap 7] {S_TOK['tokens_response'][:12]}",
            "map token -> index; token di luar vocab dapat index sementara >= vocab_size (untuk CopyNet)",
            _resp_ids[:12])

if _resp_oov:
    print(f"\n  vocab_size={_V}. OOV pada response sample (di-copy):")
    for _t, _ix in zip(S_TOK["tokens_response"], _resp_ids):
        if _ix >= _V:
            print(f"     {_t!r} -> index {_ix}  (= vocab_size + {_ix - _V})")
else:
    print(f"\n  vocab_size={_V}. OOV pada response sample: (tidak ada; semua token < vocab_size)")

# EKSPOR Tahap 9 (FLAT: index CopyNet per sample)
export_splits("09_indexed", {"train": train_indexed, "val": val_indexed, "test": test_indexed})


[SAMPLE dialog #520 turn 1] Tahap 9 - Word ke index (response)
INPUT  : [dari Tahap 7] ['<sos>', 'NAME_SLOT', 'is', 'an', 'FOOD_SLOT', 'restaurant', 'in', 'the', 'area', 'of', 'AREA_SLOT', 'in']
PROSES : map token -> index; token di luar vocab dapat index sementara >= vocab_size (untuk CopyNet)
OUTPUT : [1, 8, 18, 75, 12, 32, 21, 17, 66, 24, 13, 21]

  vocab_size=613. OOV pada response sample: (tidak ada; semua token < vocab_size)
  [EXPORT] 09_indexed: train->09_indexed_train.json, val->09_indexed_val.json, test->09_indexed_test.json


{'train': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/09_indexed_train.json'),
 'val': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/09_indexed_val.json'),
 'test': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/09_indexed_test.json')}

## 10. Padding

Samakan panjang semua sequence per split dengan menambah `<pad>`(0) di belakang.

Sekaligus **shifting** teacher forcing: `*_input` (tanpa token akhir) disodorkan ke decoder, `*_target` (tanpa token awal) untuk hitung loss — decoder belajar memprediksi token berikutnya.

In [178]:
def pad_sequences(list_of_indices, pad_idx):
    max_len = max(len(seq) for seq in list_of_indices)
    padded = [seq + [pad_idx] * (max_len - len(seq)) for seq in list_of_indices]
    return padded, max_len


def pad_dataset(indexed_samples, pad_idx):
    input_padded, input_maxlen = pad_sequences(
        [s["input_indices"] for s in indexed_samples], pad_idx
    )
    bspan_input_padded, bspan_in_maxlen = pad_sequences(
        [s["bspan_indices"][:-1] for s in indexed_samples], pad_idx
    )
    bspan_target_padded, bspan_tgt_maxlen = pad_sequences(
        [s["bspan_indices"][1:] for s in indexed_samples], pad_idx
    )
    response_input_padded, resp_in_maxlen = pad_sequences(
        [s["response_indices"][:-1] for s in indexed_samples], pad_idx
    )
    response_target_padded, resp_tgt_maxlen = pad_sequences(
        [s["response_indices"][1:] for s in indexed_samples], pad_idx
    )

    return {
        "input_padded": input_padded,
        "bspan_input_padded": bspan_input_padded,
        "bspan_target_padded": bspan_target_padded,
        "response_input_padded": response_input_padded,
        "response_target_padded": response_target_padded,
        "max_lengths": {
            "input": input_maxlen,
            "bspan_input": bspan_in_maxlen,
            "bspan_target": bspan_tgt_maxlen,
            "response_input": resp_in_maxlen,
            "response_target": resp_tgt_maxlen,
        },
    }


train_padded = pad_dataset(train_indexed, word2idx[PAD_TOKEN])
val_padded = pad_dataset(val_indexed, word2idx[PAD_TOKEN])
test_padded = pad_dataset(test_indexed, word2idx[PAD_TOKEN])

In [179]:
# Tahap 10 - Padding  (INPUT = output Tahap 9: response_indices)
_ml = train_padded["max_lengths"]["response_target"]
_rt = S_IDX["response_indices"][1:]  # response_target = tanpa <sos>
S_RESP_PAD = _rt + [word2idx[PAD_TOKEN]] * (_ml - len(_rt))
demo_sample("Tahap 10 - Padding (response_target)",
            f"[dari Tahap 9] response_target panjang {len(_rt)} -> {_rt}",
            f"tambah <pad>(0) sampai seragam = max_len {_ml}",
            f"{S_RESP_PAD[:26]} ...  (panjang {len(S_RESP_PAD)})")

# EKSPOR Tahap 10 (FLAT: matriks padded + max_lengths per split)
export_splits("10_padded", {"train": train_padded, "val": val_padded, "test": test_padded})


[SAMPLE dialog #520 turn 1] Tahap 10 - Padding (response_target)
INPUT  : [dari Tahap 9] response_target panjang 17 -> [8, 18, 75, 12, 32, 21, 17, 66, 24, 13, 21, 17, 14, 44, 45, 15, 2]
PROSES : tambah <pad>(0) sampai seragam = max_len 50
OUTPUT : [8, 18, 75, 12, 32, 21, 17, 66, 24, 13, 21, 17, 14, 44, 45, 15, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0] ...  (panjang 50)
  [EXPORT] 10_padded: train->10_padded_train.json, val->10_padded_val.json, test->10_padded_test.json


{'train': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/10_padded_train.json'),
 'val': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/10_padded_val.json'),
 'test': WindowsPath('e:/coding bebas/restorant-asistent/data/processed/eksp5/10_padded_test.json')}

## 11. Konversi ke Tensor

Bungkus data yang sudah dipadding jadi `torch.Tensor` (dtype long) — bentuk final `[jumlah_sample, max_len]` yang siap masuk model.

In [180]:
def to_tensor_dataset(padded_dict):
    return {
        "input_tensor": torch.tensor(padded_dict["input_padded"], dtype=torch.long),
        "bspan_input_tensor": torch.tensor(padded_dict["bspan_input_padded"], dtype=torch.long),
        "bspan_target_tensor": torch.tensor(padded_dict["bspan_target_padded"], dtype=torch.long),
        "response_input_tensor": torch.tensor(padded_dict["response_input_padded"], dtype=torch.long),
        "response_target_tensor": torch.tensor(padded_dict["response_target_padded"], dtype=torch.long),
    }


train_tensor = to_tensor_dataset(train_padded)
val_tensor = to_tensor_dataset(val_padded)
test_tensor = to_tensor_dataset(test_padded)

print("\nHasil konversi ke torch.Tensor:")
for name, tensor_dict in [("Train", train_tensor), ("Val", val_tensor), ("Test", test_tensor)]:
    print(f"\n{name}:")
    for k, v in tensor_dict.items():
        print(f"  {k:25s}: shape={tuple(v.shape)}, dtype={v.dtype}")


Hasil konversi ke torch.Tensor:

Train:
  input_tensor             : shape=(1635, 70), dtype=torch.int64
  bspan_input_tensor       : shape=(1635, 15), dtype=torch.int64
  bspan_target_tensor      : shape=(1635, 15), dtype=torch.int64
  response_input_tensor    : shape=(1635, 50), dtype=torch.int64
  response_target_tensor   : shape=(1635, 50), dtype=torch.int64

Val:
  input_tensor             : shape=(553, 61), dtype=torch.int64
  bspan_input_tensor       : shape=(553, 15), dtype=torch.int64
  bspan_target_tensor      : shape=(553, 15), dtype=torch.int64
  response_input_tensor    : shape=(553, 42), dtype=torch.int64
  response_target_tensor   : shape=(553, 42), dtype=torch.int64

Test:
  input_tensor             : shape=(556, 72), dtype=torch.int64
  bspan_input_tensor       : shape=(556, 13), dtype=torch.int64
  bspan_target_tensor      : shape=(556, 13), dtype=torch.int64
  response_input_tensor    : shape=(556, 43), dtype=torch.int64
  response_target_tensor   : shape=(556, 43),

In [181]:
# Tahap 11 - Konversi ke Tensor  (INPUT = output Tahap 10: padded response_target)
S_TENSOR = torch.tensor([S_RESP_PAD], dtype=torch.long)
demo_sample("Tahap 11 - Konversi ke Tensor (response_target)",
            f"[dari Tahap 10] list int panjang {len(S_RESP_PAD)}",
            "bungkus jadi torch.Tensor(dtype=long) agar bisa diproses PyTorch",
            f"shape={tuple(S_TENSOR.shape)}, dtype={S_TENSOR.dtype} -> {S_TENSOR[0][:18].tolist()} ...")

# EKSPOR Tahap 11 (tensor -> format .pt, 3 file train/val/test)
for _name, _td in [("train", train_tensor), ("val", val_tensor), ("test", test_tensor)]:
    _p = PROCESSED_DIR / f"11_tensor_{_name}.pt"
    torch.save(_td, _p)
    print(f"  [EXPORT] 11_tensor: {_name} -> {_p.name}")


[SAMPLE dialog #520 turn 1] Tahap 11 - Konversi ke Tensor (response_target)
INPUT  : [dari Tahap 10] list int panjang 50
PROSES : bungkus jadi torch.Tensor(dtype=long) agar bisa diproses PyTorch
OUTPUT : shape=(1, 50), dtype=torch.int64 -> [8, 18, 75, 12, 32, 21, 17, 66, 24, 13, 21, 17, 14, 44, 45, 15, 2, 0] ...
  [EXPORT] 11_tensor: train -> 11_tensor_train.pt
  [EXPORT] 11_tensor: val -> 11_tensor_val.pt
  [EXPORT] 11_tensor: test -> 11_tensor_test.pt
